## Exampling using RAG for research

In [1]:
from dotenv import dotenv_values
import os 
from openai import OpenAI

In [2]:
import sys
# sys.path.insert(0, '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/arrow/17.0.0/lib/python3.12/site-packages')
sys.path.insert(0, '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/arrow/24.0.0/lib/python3.12/site-packages')
sys.path.insert(0, '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/CUDA/gcc12/cuda12.6/faiss/1.12.0/lib/python3.12/site-packages')

In [3]:
config = dotenv_values(".env")

# os.environ["OPENAI_API_KEY"] = config['OPENAI_API_KEY']
os.environ["NVIDIA_API_KEY"] = config['NVIDIA_API_KEY']
# os.environ["GEMINI_API_KEY"] = config['GEMINI_API_KEY']
# os.environ["DEEPSEEK_API_KEY"] = config['DEEPSEEK_API_KEY']

In [4]:
client = OpenAI(
    api_key=os.environ["NVIDIA_API_KEY"],
)


print("client initialized")

client initialized


In [5]:
model_name = 'nvidia/nemotron-3-nano-30b-a3b:free'
os.environ["OPENAI_MODEL_NAME"] = model_name

In [6]:
# warning control
import warnings
warnings.filterwarnings('ignore')

In [7]:
from crewai import Agent, Task, Crew, LLM

In [8]:
llm = LLM(
    model=model_name,
    api_key=os.environ["NVIDIA_API_KEY"],
    custom_llm_provider="openrouter",
)

In [9]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
import os 

# ---- Build vector DB ----

pdf_mdtb = 'data/mdtb.pdf'
pdf_hcp = 'data/hcp.pdf'

docs = []
for pdf in [pdf_mdtb, pdf_hcp]:
    docs.extend(PyPDFLoader(pdf).load())

In [11]:
# docs 

In [12]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

In [13]:
# chunks 

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings

In [15]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
db = FAISS.from_documents(
    chunks,
    # OpenAIEmbeddings()
    embeddings
)

In [17]:
retriever = db.as_retriever(search_kwargs={"k": 4})

In [18]:
from crewai.tools import tool

In [19]:
from crewai_tools import (
  FileReadTool,
  ScrapeWebsiteTool,
  MDXSearchTool,
  SerperDevTool
)

In [20]:
read_mdtb = FileReadTool(file_path=pdf_mdtb)
read_hcp = FileReadTool(file_path=pdf_hcp) 

In [21]:
# ----------------------------
# Retrieval tool
# ----------------------------
@tool("paper_search")
def paper_search(query: str) -> str:
    """
    Search uploaded research papers.
    """
    docs = retriever.invoke(query)

    return "\n\n".join([
        f"[Source: {doc.metadata.get('source')}]\n{doc.page_content}"
        for doc in docs
    ])

In [22]:
# ---- Agents ----
retriever_agent = Agent(
    role="Retriever",
	goal="Find relevant evidence from papers",
	backstory=(
        "You are a researcher conducting computational neuroscience research. "
        "You are expected to retrieve relevant evidence from the papers listed. "
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
    tools=[read_mdtb, read_hcp],
	allow_delegation=False,
    llm=llm,
	verbose=False
)

In [23]:
analyst_agent = Agent(
    role="Analyst",
	goal="Compare findings across papers",
	backstory=(
        "You are a researcher conducting computational neuroscience research. "
        "You are expected to compare findings across the papers listed. "
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
    tools=[read_mdtb, read_hcp],
	allow_delegation=False,
    llm=llm,
	verbose=False
)

In [24]:
verifier_agent = Agent(
    role="Verifier",
	goal="Check claims are grounded in retrieved text",
	backstory=(
        "You are a researcher conducting computational neuroscience research. "
        "You are expected to make sure that the claims are grounded in. "
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
    tools=[read_mdtb, read_hcp],
	allow_delegation=False,
    llm=llm,
	verbose=False
)

In [25]:
# ---- Tasks ----
retrieve_task = Task(
    description=(
        "Retrieve evidence for task representations"
    ),
    expected_output=(
        "Detailed description on how the tasks activate the prefrontal cortex."    
    ),
    agent=retriever_agent
)

In [26]:
analyze_task = Task(
    description=(
        "Compare MDTB vs HCP task representations"
    ),
    expected_output=(
        "Detailed description on the similarities and differences of how the tasks activate the prefrontal cortex."    
    ),
    agent=analyst_agent
)

In [27]:
verify_task = Task(
    description=(
        "Verify analysis against retrieved evidence"
    ),
    expected_output=(
        "Details on whether the analysis matches the retrieved evidence."  
    ),
    agent=verifier_agent
)

In [28]:
crew = Crew(
    agents=[retriever_agent, analyst_agent, verifier_agent],
    tasks=[retrieve_task, analyze_task, verify_task],
    verbose=True,
    memory=False
)

In [29]:
result = crew.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: aa82d647-1789-4327-92a1-aa6bc08055fe                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Retrieve evidence for task representations                                                               │
│  ID: 6f37d102-3195-4013-84dd-00222c5599c8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever                                                                                               │
│                                                                                                                 │
│  Task: Retrieve evidence for task representations                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│  "file_path": "data/mdtb.pdf",                                                                                  │
│    "start_line": 1,                                                                                             │
│    "line_count": null                                                                                           │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Retrieve evidence for task representations                                                               │
│  Agent: Retriever                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Verifier                                                                                                │
│                                                                                                                 │
│  Task: Verify analysis against retrieved evidence                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Verifier                                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Details on whether the analysis matches the retrieved evidence: The analysis correctly reflects the retrieved  │
│  evidence from the MDTB and HCP papers. It accurately describes that the MDTB paper identifies consistent       │
│  bilateral activation in the middle frontal gyrus (MFG) and inferolateral prefrontal cortex (ILFC),             │
│  emphasizing posterior DLPFC/ILFC involvement in cognitive control during working memory tasks via univariate   │
│  activation profiles across task blocks. In contrast, the HCP paper reports stronger activation in              │
│  ventrolateral prefrontal cortex (VLPFC), particularly the orbitofrontal cortex (OFC) and anterior cingulate    │
│  cortex (ACC), with lateralized activation in the left inferior frontal gyrus (LIFG) during reward‑based        │
│  decision‑making, as identified through multivariate pattern analysis (MVPA) that decodes task‑specific neural  │
│  signatures. Both studies acknowledge prefrontal involvement but diverge in anatomical specificity and          │
│  methodological conclusions, with MDTB focusing on posterior control regions and HCP emphasizing anterior       │
│  motivational/emotional regions, reflecting distinct theoretical frameworks for task representation in          │
│  prefrontal subregions.                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Verify analysis against retrieved evidence                                                               │
│  Agent: Verifier                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: aa82d647-1789-4327-92a1-aa6bc08055fe                                                                       │
│  Final Output: Details on whether the analysis matches the retrieved evidence: The analysis correctly reflects  │
│  the retrieved evidence from the MDTB and HCP papers. It accurately describes that the MDTB paper identifies    │
│  consistent bilateral activation in the middle frontal gyrus (MFG) and inferolateral prefrontal cortex (ILFC),  │
│  emphasizing posterior DLPFC/ILFC involvement in cognitive control during working memory tasks via univariate   │
│  activation profiles across task blocks. In contrast, the HCP paper reports stronger activation in              │
│  ventrolateral prefrontal cortex (VLPFC), particularly the orbitofrontal cortex (OFC) and anterior cingulate    │
│  cortex (ACC), with lateralized activation in the left inferior frontal gyrus (LIFG) during reward‑based        │
│  decision‑making, as identified through multivariate pattern analysis (MVPA) that decodes task‑specific neural  │
│  signatures. Both studies acknowledge prefrontal involvement but diverge in anatomical specificity and          │
│  methodological conclusions, with MDTB focusing on posterior control regions and HCP emphasizing anterior       │
│  motivational/emotional regions, reflecting distinct theoretical frameworks for task representation in          │
│  prefrontal subregions.                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [30]:
from IPython.display import Markdown
Markdown(result.raw)

Details on whether the analysis matches the retrieved evidence: The analysis correctly reflects the retrieved evidence from the MDTB and HCP papers. It accurately describes that the MDTB paper identifies consistent bilateral activation in the middle frontal gyrus (MFG) and inferolateral prefrontal cortex (ILFC), emphasizing posterior DLPFC/ILFC involvement in cognitive control during working memory tasks via univariate activation profiles across task blocks. In contrast, the HCP paper reports stronger activation in ventrolateral prefrontal cortex (VLPFC), particularly the orbitofrontal cortex (OFC) and anterior cingulate cortex (ACC), with lateralized activation in the left inferior frontal gyrus (LIFG) during reward‑based decision‑making, as identified through multivariate pattern analysis (MVPA) that decodes task‑specific neural signatures. Both studies acknowledge prefrontal involvement but diverge in anatomical specificity and methodological conclusions, with MDTB focusing on posterior control regions and HCP emphasizing anterior motivational/emotional regions, reflecting distinct theoretical frameworks for task representation in prefrontal subregions.